In [86]:
import os
import json
import re
from typing import List, Optional
from dotenv import load_dotenv
from langchain_groq import ChatGroq

load_dotenv()

model = ChatGroq(model ="llama-3.3-70b-versatile",
                 api_key=os.getenv("GROQ_API_KEY"),
                 temperature = 0)



In [87]:
def extract_json(text):
    match = re.search(r'\{.*\}', text, re.DOTALL)
    return match.group(0) if match else text

def country_pref(value):
    if not value:
        return ""
    value = value.lower()
    if "outside" in value or "foreign" in value or "international" in value or "non" in value or "not" in value:
        return "foreign"
    if "usa" in value or "us" in value or "america" in value:
        return "us"
    return ""

def is_invalid_output(data):
    return data.get("invalid") == True


In [88]:
def build_url(filters):
    base_url = "https://planefax.com/aircraft/for-sale/"
    params = []

    if filters.get("cat"):
        for c in filters["cat"]:
            params.append(f"cat={c}")

    if filters.get("manufacturer"):
        for m in filters["manufacturer"]:
            params.append(f"manufacturer={m}")

    params.append(f"year__gte={filters.get('year__gte') or ''}")
    params.append(f"year__lte={filters.get('year__lte') or ''}")
    params.append(f"min_price={filters.get('min_price') or ''}")
    params.append(f"max_price={filters.get('max_price') or ''}")

    if filters.get("state"):
        for s in filters["state"]:
            params.append(f"state={s}")

    country = country_pref(filters.get("country_pref"))
    params.append(f"country_pref={country}")

    params.append(f"min_date={filters.get('min_date') or ''}")
    params.append(f"max_date={filters.get('max_date') or ''}")

    return base_url + "?" + "&".join(params)


In [ ]:
def chatbot(query):
    prompt = f"""
You are a filter extraction AI. Convert the user query into ONLY valid JSON. 
Do not include any explanation, text, or markdown.

Return JSON with this structure:
{{
"cat": [],
"manufacturer": [],
"year__gte": null,
"year__lte": null,
"min_price": null,
"max_price": null,
"state": [],
"country_pref": null,
"min_date": null,
"max_date": null
}}

Rules:

ONLY JSON output (no text)
manufacturer must be UPPERCASE
state must be valid US state codes (e.g., CA, TX, UT)
country_pref must be either "us" or "foreign"

Number and range rules:

"before 2019" → year__lte = 2018
"after 2015" → year__gte = 2016
"1 million" → 1000000

Text normalization rules:

Treat user input as noisy and possibly misspelled
Always correct spelling before extracting filters
Fix aircraft names and common typos
Treat any variation of USA (USA, US, U.S.A, USAd, usd when used as country) as "us"

State filter rules:

Only include "state" if user explicitly mentions specific states
If user says "USA", "US", or "all US", DO NOT include any state filters
If country_pref is "us", state must be empty []

Category IDs:

Jet Aircraft = 1
Turboprop Aircraft = 2
Piston Single Aircraft = 3
Piston Twin Aircraft = 4
Piston Helicopters = 13
Turbine Helicopters = 14

You MUST only process queries related to:
- airplanes
- aircraft
- jets
- helicopters
- aviation
- aircraft manufacturers

You must determine if the user query is a SEARCH query or NOT.
A valid SEARCH query:
- asks to show, list, find, or filter aircraft
- implies browsing inventory
Examples:
- "show jets under 1 million"
- "planes in texas"
- "list cessna aircraft"

An INVALID query:
- asks for definition, explanation, or general knowledge
- is not about filtering or listing aircraft

Examples:
- "how do planes fly"
- "who invented aircraft"
- "hello"
- "random text"

If the user query is NOT related to aircraft/aviation or it is not a SEARCH query,
- Return ONLY this JSON:
{{ "invalid": true }}

Do NOT extract filters.
Do NOT guess intent.
Do NOT respond to unrelated topics.

User query:
{query}
"""

    response = model.invoke(prompt)

    json_text = extract_json(response.content)
    try:
        data = json.loads(json_text)
    except Exception as e:
        print("JSON parsing failed")
        print("Raw output:", response.content)
        return None
    
    if is_invalid_output(data):
        return "Invalid input: I can only help with aircraft searches."

    return build_url(data)


In [99]:
user_input = input("Apply filters for aircraft search: ")
print(chatbot(user_input))


Invalid input: only aircraft-related queries are allowed.
